In [1]:
#Imports
import pickle
import os
from pathlib import Path
from shared.utils import run_model, load_snapshot
from shared.plotting import plot_sankey
from energyscope.models import Model

# 2050 - Sans pointe

In [2]:
energyscope_original_snapshot_2050 = load_snapshot(year=2050, scenario=True)
results_wo_peaks_2050 = run_model(energyscope_original_snapshot_2050,apply_postprocessing=True)
plot_sankey(results_wo_peaks_2050)

	C:\Users\julie\Desktop\EnergyScope-Quebec\shared\scenarios\QC_generic_scenario_2050.dat
	line 5 offset 137
	f_max needs 2 subscripts, not 1
	context:  let  >>> f_max[f] <<<  := 0 ;
Gurobi 12.0.3: 

In [17]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_wo_peaks_2050.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_wo_peaks_2050, f)

# 2050 - Sans pointe - Updates

In [95]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_UTILITIES_DIR = _BASE_DIR / 'shared' / 'utilities' / 'ES_snapshot'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle unique ordonné
energyscope_original_snapshot_2050_update = Model([
    # --- FICHIERS DE BASE (Partagés) ---
    ('mod', str(_UTILITIES_DIR / 'QC_es_main.mod')),
    ('mod', str(_UTILITIES_DIR / 'QC_objective_function.mod')),
    ('dat', str(_UTILITIES_DIR / 'QC_data.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_techs_dist_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_techs_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_params.dat')),

    # --- ÉLÉMENTS DE MODÉLISATION SUPPLÉMENTAIRES (.mod) ---
    ('mod', str(_PROJECT_DIR / 'HP_extra.mod')),
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),
    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')),
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),
    ('mod', str(_PROJECT_DIR / 'realisme_2050.mod')),     # <-- REPLACÉ ICI : Déclaration propre sans "model;"

    # --- DONNÉES ET SCRIPTS DE CONTEXTE ---
    # Flag 'mod' pour basculer AMPL en mode script juste avant les données HP
    ('mod', str(_PROJECT_DIR / 'HP_extra.dat')),

    # Injection des données des pompes à chaleur et autres imports
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.dat')),

    # Injection des données de réalisme
    #('dat', str(_PROJECT_DIR / 'realisme_2050.dat')),     # <-- Injection des listes explicites
])

In [96]:
results_wo_peaks_2050_update = run_model(energyscope_original_snapshot_2050_update,apply_postprocessing=True)
plot_sankey(results_wo_peaks_2050_update)

	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.mod
	line 4 offset 243
	YEARS is not defined
	context:  subject to link_hp_normal_capacity {y in  >>> YEARS} <<< :
	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\realisme_2050.mod
	line 9 offset 416
	YEARS is not defined
	context:  param share_suv_min  >>> {YEARS} <<<  >= 0, <= 1 default 0.30;
	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.dat
	line 15 offset 598
	YEARS is not defined
	context:  let {y in  >>> YEARS} <<<  f_max[y, 'DEC_HP_ELEC_WINTER'] := f_max[y, 'DEC_HP_ELEC'];
Gurobi 12.0.3: 

In [97]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_wo_peaks_2050_update.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_wo_peaks_2050_update, f)

In [98]:
# Extraction de la série / dataframe de la variable
F_Mult_t_2050 = results_wo_peaks_2050_update.variables['F_Mult_t']

# Filtrage sur le niveau "TECHNOLOGIES" (ou le niveau 1)
df_suv = F_Mult_t_2050[F_Mult_t_2050.index.get_level_values(0).str.startswith('SUV')]
df_suv.sum()

F_Mult_t    0.0
Run         0.0
dtype: float64

In [93]:
F_Mult_2050 = results_wo_peaks_2050_update.variables['F_Mult']
# Sélection des lignes dont l'index commence par 'SUC'
df_suv = F_Mult_2050[F_Mult_2050.index.str.startswith('SUV')]

# Affichage du résultat
print(df_suv.sum())

F_Mult    0.0
Run       0.0
dtype: float64


In [94]:
# 1. Récupération de la variable (qui n'a plus de dimension 'YEARS')
F_Mult_t_snapshot = results_wo_peaks_2050_update.variables['F_Mult_t']

# 2. Filtrage des SUV (le niveau 0 est maintenant celui des technologies)
tech_index = F_Mult_t_snapshot.index.get_level_values(0).str.upper()
df_suv = F_Mult_t_snapshot[tech_index.str.startswith('SUV')]

# 3. On élimine les 0 et les NaN
df_suv_actifs = df_suv[df_suv.iloc[:, 0] > 1e-4].dropna()

# Affichage du résultat
if df_suv_actifs.empty:
    print("⚠️ Toujours à 0. Vérifiez si les f_max des SUV ne sont pas bloqués à 0 dans vos fichiers de données de base.")
else:
    print("🎉 Succès ! Voici vos SUV actifs pour ce snapshot :")
    print(df_suv_actifs)

⚠️ Toujours à 0. Vérifiez si les f_max des SUV ne sont pas bloqués à 0 dans vos fichiers de données de base.


# 2050 - Sans pointe - Carboneutre

In [26]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_UTILITIES_DIR = _BASE_DIR / 'shared' / 'utilities' / 'ES_snapshot'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle unique ordonné (2050 Carboneutre - Sans Pointes, Avec HP)
energyscope_original_snapshot_2050_carboneutre = Model([
    # --- FICHIERS DE BASE (Partagés) ---
    ('mod', str(_UTILITIES_DIR / 'QC_es_main.mod')),
    ('mod', str(_UTILITIES_DIR / 'QC_objective_function.mod')),
    ('dat', str(_UTILITIES_DIR / 'QC_data.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_techs_dist_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_techs_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_params.dat')),

    # --- ÉLÉMENTS DE MODÉLISATION SUPPLÉMENTAIRES (.mod) ---
    ('mod', str(_PROJECT_DIR / 'HP_extra.mod')),     # Déclarations / structures pour les HP
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),    # Logique hivernale des HP
    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')), # Limite annuelle d'hydro
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),

    # --- DONNÉES ET SCRIPTS DE CONTEXTE ---
    # 1. Données spécifiques au scénario carboneutre (ex: co2_limit := 0;)
    ('dat', str(_PROJECT_DIR / 'carboneutre.dat')),

    # 2. Flag 'mod' pour basculer AMPL en mode script juste avant les données HP
    ('mod', str(_PROJECT_DIR / 'HP_extra.dat')),     # Ton fichier (vide ou avec commentaires) qui ouvre les droits du 'let'

    # 3. Injection des données des pompes à chaleur
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),     # Profite du mode script pour exécuter son 'let cop_normal'
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.dat')),
])

In [27]:
results_wo_peaks_2050_carboneutre = run_model(energyscope_original_snapshot_2050_carboneutre,apply_postprocessing=True)
plot_sankey(results_wo_peaks_2050_carboneutre)

	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.mod
	line 4 offset 243
	YEARS is not defined
	context:  subject to link_hp_normal_capacity {y in  >>> YEARS} <<< :
	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.dat
	line 15 offset 598
	YEARS is not defined
	context:  let {y in  >>> YEARS} <<<  f_max[y, 'DEC_HP_ELEC_WINTER'] := f_max[y, 'DEC_HP_ELEC'];
Gurobi 12.0.3: 

In [31]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_wo_peaks_2050_carboneutre.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_wo_peaks_2050_carboneutre, f)

# 2050 - Sans pointe - Caboneutre - Plan HQ

In [42]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_UTILITIES_DIR = _BASE_DIR / 'shared' / 'utilities' / 'ES_snapshot'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle unique ordonné (2050 Carboneutre - Sans Pointes, Avec HP)
energyscope_original_snapshot_2050_carboneutre_planHQ = Model([
    # --- FICHIERS DE BASE (Partagés) ---
    ('mod', str(_UTILITIES_DIR / 'QC_es_main.mod')),
    ('mod', str(_UTILITIES_DIR / 'QC_objective_function.mod')),
    ('mod', str(_PROJECT_DIR / 'plan_action_HQ_2035.mod')),
    ('dat', str(_UTILITIES_DIR / 'QC_data.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_techs_dist_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_techs_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_params.dat')),

    # --- ÉLÉMENTS DE MODÉLISATION SUPPLÉMENTAIRES (.mod) ---
    ('mod', str(_PROJECT_DIR / 'HP_extra.mod')),     # Déclarations / structures pour les HP
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),    # Logique hivernale des HP
    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')), # Limite annuelle d'hydro
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),

    # --- DONNÉES ET SCRIPTS DE CONTEXTE ---
    # 1. Données spécifiques au scénario carboneutre (ex: co2_limit := 0;)
    ('dat', str(_PROJECT_DIR / 'carboneutre.dat')),

    # 2. Flag 'mod' pour basculer AMPL en mode script juste avant les données HP
    ('mod', str(_PROJECT_DIR / 'HP_extra.dat')),     # Ton fichier (vide ou avec commentaires) qui ouvre les droits du 'let'

    # 3. Injection des données des pompes à chaleur
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),     # Profite du mode script pour exécuter son 'let cop_normal'
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.dat')),
])

In [43]:
results_wo_peaks_2050_carboneutre_planHQ = run_model(energyscope_original_snapshot_2050_carboneutre_planHQ,apply_postprocessing=True)
plot_sankey(results_wo_peaks_2050_carboneutre)

	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.mod
	line 4 offset 243
	YEARS is not defined
	context:  subject to link_hp_normal_capacity {y in  >>> YEARS} <<< :
	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.dat
	line 15 offset 598
	YEARS is not defined
	context:  let {y in  >>> YEARS} <<<  f_max[y, 'DEC_HP_ELEC_WINTER'] := f_max[y, 'DEC_HP_ELEC'];
Gurobi 12.0.3: 

In [44]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_wo_peaks_2050_carboneutre_planHQ.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_wo_peaks_2050_carboneutre_planHQ, f)

# 2050 - Avec pointe - Carboneutre

In [32]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_UTILITIES_DIR = _BASE_DIR / 'shared' / 'utilities' / 'ES_snapshot'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle 2050 Carboneutre natif
energyscope_peaks_2050_carboneutre = Model([
    # --- STRUCTURES MATHÉMATIQUES (.mod) ---
    ('mod', str(_UTILITIES_DIR / 'QC_es_main.mod')),
    ('mod', str(_UTILITIES_DIR / 'QC_objective_function.mod')),
    ('mod', str(_PROJECT_DIR / 'HP_extra.mod')),
    ('mod', str(_PROJECT_DIR / 'peaks_extra.mod')),  # Contient la déclaration 'set PEAK_PERIODS;'
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),
    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')), # Limite annuelle d'hydro
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),

    # --- DONNÉES DE BASE ENERGYSCOPE ---
    ('dat', str(_UTILITIES_DIR / 'QC_data.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_techs_dist_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_techs_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_params.dat')),

    # --- SCÉNARIO 2050 CARBONEUTRE ---
    ('dat', str(_PROJECT_DIR / 'carboneutre.dat')),   # param co2_limit := 0;

    # --- ENCLENCHEMENT DU MODE POINTES (La feinte du 'let') ---
    # 1. Initialisation du set avec les périodes 13 et 14
    ('dat', str(_PROJECT_DIR / 'peaks_2_periods.dat')),

    # 2. Flag 'mod' pour basculer AMPL en mode script (fusionne {13, 14} dans PERIODS)
    ('mod', str(_PROJECT_DIR / 'peaks_extra.dat')),

    # 3. Injection des données 2050 (AMPL acceptera le 'let' à la ligne 4 car il est en mode script)
    ('dat', str(_PROJECT_DIR / 'peaks_2_2050.dat')),
    ('dat', str(_PROJECT_DIR / 'peaks_cpt.dat')),
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.dat')),
])

In [33]:
results_peaks_2050_carboneutre = run_model(energyscope_peaks_2050_carboneutre,apply_postprocessing=True)
plot_sankey(results_peaks_2050_carboneutre)

	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.mod
	line 4 offset 243
	YEARS is not defined
	context:  subject to link_hp_normal_capacity {y in  >>> YEARS} <<< :
	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.dat
	line 15 offset 598
	YEARS is not defined
	context:  let {y in  >>> YEARS} <<<  f_max[y, 'DEC_HP_ELEC_WINTER'] := f_max[y, 'DEC_HP_ELEC'];
Gurobi 12.0.3: 

In [34]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_peaks_2050_carboneutre.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_peaks_2050_carboneutre, f)

# 2050 - Avec pointe - Carboneutre - Plan HQ

In [46]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_UTILITIES_DIR = _BASE_DIR / 'shared' / 'utilities' / 'ES_snapshot'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle 2050 Carboneutre natif
energyscope_peaks_2050_carboneutre_planHQ = Model([
    # --- STRUCTURES MATHÉMATIQUES (.mod) ---
    ('mod', str(_UTILITIES_DIR / 'QC_es_main.mod')),
    ('mod', str(_UTILITIES_DIR / 'QC_objective_function.mod')),
    ('mod', str(_PROJECT_DIR / 'HP_extra.mod')),
    ('mod', str(_PROJECT_DIR / 'peaks_extra.mod')),  # Contient la déclaration 'set PEAK_PERIODS;'
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),
    ('mod', str(_PROJECT_DIR / 'plan_action_HQ_2035.mod')),
    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')), # Limite annuelle d'hydro
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),

    # --- DONNÉES DE BASE ENERGYSCOPE ---
    ('dat', str(_UTILITIES_DIR / 'QC_data.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_techs_dist_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_techs_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_params.dat')),

    # --- SCÉNARIO 2050 CARBONEUTRE ---
    ('dat', str(_PROJECT_DIR / 'carboneutre.dat')),   # param co2_limit := 0;

    # --- ENCLENCHEMENT DU MODE POINTES (La feinte du 'let') ---
    # 1. Initialisation du set avec les périodes 13 et 14
    ('dat', str(_PROJECT_DIR / 'peaks_2_periods.dat')),

    # 2. Flag 'mod' pour basculer AMPL en mode script (fusionne {13, 14} dans PERIODS)
    ('mod', str(_PROJECT_DIR / 'peaks_extra.dat')),

    # 3. Injection des données 2050 (AMPL acceptera le 'let' à la ligne 4 car il est en mode script)
    ('dat', str(_PROJECT_DIR / 'peaks_2_2050.dat')),
    ('dat', str(_PROJECT_DIR / 'peaks_cpt.dat')),
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.dat')),

])

In [47]:
results_peaks_2050_carboneutre_planHQ = run_model(energyscope_peaks_2050_carboneutre_planHQ,apply_postprocessing=True)
plot_sankey(results_peaks_2050_carboneutre_planHQ)

	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.mod
	line 4 offset 243
	YEARS is not defined
	context:  subject to link_hp_normal_capacity {y in  >>> YEARS} <<< :
	C:\Users\julie\Desktop\EnergyScope-Quebec\projects\peaks\02_Model\HP_Winter.dat
	line 15 offset 598
	YEARS is not defined
	context:  let {y in  >>> YEARS} <<<  f_max[y, 'DEC_HP_ELEC_WINTER'] := f_max[y, 'DEC_HP_ELEC'];
Gurobi 12.0.3: 

In [49]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_peaks_2050_carboneutre_planHQ.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_peaks_2050_carboneutre_planHQ, f)